1. 待评估的模型列表

In [1]:
# {"model_path": "note"}
model_list = {
    "sft_mini_llama3": "sft baseline",                            # sft baseline
    "dpo_mini_llama3": "dpo",                                     # dpo
    "dpo_mini_llama3_1": "dpo + sft warmup",                      # dpo + sft warmup 
    "dpo_mini_llama3_2": "dpo + freeze 8 layers",                 # dpo + freeze 8 layers
    "dpo_mini_llama3_3": "dpo + sft warmup + freeze 8 layers",    # dpo + sft warmup + freeze 8 layers
    "dpop_mini_llama3": "dpop",                                   # dpop
    "dpop_mini_llama3_1": "dpop + sft warmup",                    # dpop + sft warmup
    "dpop_mini_llama3_2": "dpop + freeze 8 layers",               # dpop + freeze 8 layers
    "dpop_mini_llama3_3": "dpop + sft warmup + freeze 8 layers",  # dpop + sft warmup + freeze 8 layers
    "dpop_mini_llama3_4": "dpop + sft warmup (more aggressive)",  # dpop + sft warmup, 更加激进的设置: max_lr=1e-5, min_lr=1e-6, beta=0.5
}

2. 训练结果
 
除非特殊说明，一下结果主要采用如下配置训练：

```python
max_seq_len = 512
max_batch_size = 12
min_score = 3
max_score = 5
epochs=1
max_lr = 1e-6
min_lr = 1e-7
beta = 0.1
```

以下是不同方法的训练结果曲线（不一定最优，仅供参考）：

- dpo

<div align="center">
<img src="../assets/dpo.png" width="50%" alt="dpo">
</div>

- dpo + sft warmup

<div align="center">
<img src="../assets/dpo+sft_warmup.png" width="50%" alt="dpo_sft">
</div>

- dpo + freeze 8 layers

<div align="center">
<img src="../assets/dpo+freeze_8_layers.png" width="50%" alt="dpo_freeze_8_layers">
</div>

- dpo + sft warmup + freeze 8 layers

<div align="center">
<img src="../assets/dpo+sft_warmup+freeze_8_layers.png" width="50%" alt="dpo_sft_freeze_8_layers">
</div>

- dpop

<div align="center">
<img src="../assets/dpop.png" width="50%" alt="dpop">
</div>

- dpop + sft warmup

<div align="center">
<img src="../assets/dpop+sft_warmup.png" width="50%" alt="dpop_sft">
</div>

- dpop + freeze 8 layers

<div align="center">
<img src="../assets/dpop+freeze_8_layers.png" width="50%" alt="dpop_freeze_8_layers">
</div>

- dpop + sft warmup + freeze 8 layers

<div align="center">
<img src="../assets/dpop+sft_warmup+freeze_8_layers.png" width="50%" alt="dpop_sft_freeze_8_layers">
</div>


仅根据以上训练曲线来看，大致粗浅分析如下：

- 对于本项目这种小模型而言，纯 dpo 需要结合较小的学习率和 $\beta$，否则很容易训崩
- 在默认筛选的 3-5 分数据集中，模型已经具备一定的区分能力，这反应在训练初始时，chosen 的 logp 就已经比 rejected 的 logp 稍大
- 由于使用的是公开数据集，数据并不是从已有的模型采样得到的，因此预先对 policy 模型在 chosen 数据上进行 sft warmup，可以减小分部偏差，相比直接使用 dpo，sft warmup 后的 chosen logp 会稍大一些
- 为了防止小模型训崩，可以选择冻结一部分参数，不过这也意味着训练更加保守
- dpop 训练过程中，单看 loss 可能下降并不明显，因为它可能会受到 chosen logp 下降带来的惩罚，但 reward margin 还是有提高的

基于以上结果，选择 dpop + sft warmup，配置**更加激进**的参数：

```python
max_lr = 1e-5
min_lr = 1e-6
beta = 0.5
```

得到如下训练结果：

- dpop + sft warmup (more aggressive)

<div align="center">
<img src="../assets/dpop+sft_warmup_more_aggressive.png" width="50%" alt="dpop_sft_more_aggressive">
</div>

可见，仅从训练曲线来看，在此设置下的结果还是不错的（相同配置下的纯 dpo 会使小模型训崩，这里未展示其训练曲线）。

3. 不同模型的生成结果示例

In [2]:
import mini_models  # 先导入用于注册自定义的 Mini-LLM 模型
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [9]:
tokenizer = AutoTokenizer.from_pretrained("../mini_tokenizer")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def generate_response(question):
    print("=" * 50)
    print("Question: ", question)
    print("=" * 50)
    for model_name in list(model_list.keys()):
        model = AutoModelForCausalLM.from_pretrained(f"../output/{model_name}")
        model = model.to(device)
        
        messages = [{"role": "user", "content": f"{question}"}]
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = tokenizer(formatted_text, return_tensors="pt")["input_ids"].to(model.device)
        response = model.generate(input_ids, max_new_tokens=400)
        response = tokenizer.decode(response[0][len(input_ids[0]):], skip_special_tokens=True)
        print("Model: ", model_list[model_name])
        print(response)
        print("-" * 50)
        
        del model
        torch.cuda.empty_cache()

In [10]:
generate_response("你好，你是谁？")

Question:  你好，你是谁？
Model:  sft baseline
你好！我是Mini-LLM，一个由开发者WKQ创建的聊天机器人助手。
--------------------------------------------------
Model:  dpo
你好！我是Mini-LLM，一个由开发者WKQ创建的语言模型。我被训练来理解和生成自然语言文本，包括文本、语音和图像等。
我的设计目标是成为一个能够理解和生成各种语言的模型，包括但不限于文本生成、对话系统、机器翻译、语音识别、情感分析、文本摘要、对话系统等。我的设计基于大量的语言数据和深度学习算法，包括词向量、循环神经网络、注意力机制等，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括词向量、循环神经网络、注意力机制等，以捕捉文本中的语义和上下文信息。我的设计基于大量的语言数据和深度学习算法，包括词向量、循环神经网络、注意力机制等，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，以捕捉文本中的语义和上下文信息。
我的设计基于大量的语言数据和深度学习算法，包括大量的文本数据和语言模型训练数据，
--------------------------------------------------
Model:  dpo + sf

In [11]:
generate_response("简要介绍一下transformer模型。")

Question:  简要介绍一下transformer模型。
Model:  sft baseline
Transformer模型是一种基于注意力机制（Attention）的神经网络模型，它使用注意力机制来学习输入序列中的信息，并输出一个与输入序列相关的输出。

Transformer模型由两个部分组成：输入层（Input Layer）和输出层（Output Layer）。输入层接受输入序列，输出层则输出一个与输入序列相关的输出。

Transformer模型的核心思想是将输入序列中的每个字符视为一个向量，并将它们表示为向量之间的加权和。这个向量被称为注意力向量，它包含了输入序列中每个字符的上下文信息。

在Transformer模型中，每个输入序列都有一个注意力机制，这个机制会根据输入序列中的每个字符的上下文信息来计算一个向量，并将这个向量作为输入序列的输出。

Transformer模型通常用于自然语言处理任务，例如文本分类、情感分析、机器翻译等。它可以通过对输入序列进行编码，然后使用注意力机制来学习输入序列中的信息，并输出一个与输入序列相关的输出。

总之，Transformer模型是一种强大的神经网络模型，可以用于各种自然语言处理任务，并且具有广泛的应用前景。
--------------------------------------------------
Model:  dpo
Transformer（自注意力机制）是一种深度学习模型，用于解决序列数据中的序列建模问题。它的主要思想是将输入序列视为一个向量，通过引入注意力机制，使得模型能够更好地捕捉序列中的依赖关系，从而更好地捕捉序列中的上下文信息。
Transformer模型的核心思想是将输入序列视为一个向量，通过引入注意力机制，使得模型能够更好地捕捉序列中的依赖关系，从而更好地捕捉序列中的上下文信息。具体来说，Transformer模型的核心思想是将输入序列视为一个向量，通过引入注意力机制，使得模型能够更好地捕捉序列中的上下文信息。具体来说，Transformer模型的核心思想是将输入序列视为一个向量，通过引入注意力机制，使得模型能够更好地捕捉序列中的上下文信息。具体来说，Transformer模型的核心思想是将输入序列视为一个向量，通过引入注意力机制，使得模型能够更好地捕捉序列中的上下文信息。具

In [13]:
generate_response("天空为什么是蓝色的？")

Question:  天空为什么是蓝色的？
Model:  sft baseline
天空为什么是蓝色的，这是一个非常有趣的问题。实际上，天空之所以呈现出蓝色，是因为大气层中的气体分子对太阳光中的蓝色光进行散射。
当太阳光穿过大气层时，蓝色光被大气分子散射，其中蓝色光波长较短，因此更容易被散射。这就是为什么我们看到的天空是蓝色的原因。
当太阳光穿过大气层时，它会与大气分子中的分子发生相互作用。这些分子会散射蓝色光，使得蓝色光更容易被散射。这就是为什么我们看到的天空是蓝色的原因。
此外，大气层中的气体分子对太阳光中的蓝色光进行散射。这些散射现象被称为“散射”。散射是指光线在穿过大气层时，遇到不同的介质时，会改变其方向，从而改变其颜色。
总之，天空为什么是蓝色，是因为大气层中的气体分子对太阳光中的蓝色光进行散射。
--------------------------------------------------
Model:  dpo
天空之所以呈现蓝色，是因为大气层中的气体分子对蓝色光的散射作用。
当太阳光穿过大气层时，它会与气体分子发生散射。散射是指光线从一种方向射向另一种方向的现象。当太阳光穿过大气层时，它会与大气分子中的分子发生散射。这些分子会散射出蓝色和绿色的光线，其中包括红、橙、黄、绿、青、蓝、紫等颜色的光线。
当太阳光穿过大气层时，它会与大气分子中的分子发生散射。散射是指光线从一种方向射向另一种方向的现象。当太阳光穿过大气层时，它会与大气分子中的分子发生散射。散射的程度取决于散射光的强度。当太阳光穿过大气层时，它会与大气分子中的分子发生散射，其中包括红、橙、黄、绿、青、蓝、紫等颜色的光线。
因此，天空为什么呈现蓝色，是因为大气层中的气体分子对蓝色光的散射作用。当太阳光穿过大气层时，它会与大气分子中的分子发生散射。散射的程度取决于散射光的强度。当太阳光穿过大气层时，它会与大气分子中的分子发生散射，其中包括红、橙、黄、绿、青、蓝、紫等颜色的光线。
总之，天空呈现蓝色是由于大气层中的气体分子对蓝色光的散射作用。当太阳光穿过大气层时，它会与大气分子中的分子发生散射，其中包括红、橙、黄、绿、青、蓝、紫等颜色的光线。
--------------------------------------------------
Model:  dpo + sft warmup